In [ ]:
pip install greenlet

In [ ]:
from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types
from dotenv import load_dotenv

print("✅ ADK components imported successfully.")

load_dotenv()

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

**Retry config.** `retry_config` tells the Gemini client to automatically retry on transient failures (HTTP 429/500/503/504) with exponential backoff — up to 5 attempts, starting at 1s and multiplying by 7 each time. It gets passed into every `Gemini(...)` model instance below so agent calls don't fail on a momentary rate limit or server hiccup.

In [ ]:
# Define helper functions that will be reused throughout the notebook
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query)])

            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query
            ):
                # Check if the event contains valid content
                if event.content and event.content.parts:
                    # Filter out empty or "None" responses before printing
                    if (
                        event.content.parts[0].text != "None"
                        and event.content.parts[0].text
                    ):
                        print(f"{MODEL_NAME} > ", event.content.parts[0].text)
    else:
        print("No queries!")


print("✅ Helper functions defined.")

**The `run_session` helper.** This wraps the usual ADK flow: try to create a session with the given `session_name`, or fall back to fetching it if it already exists; then send each query through `runner_instance.run_async(...)` and print the streamed model response. Reusing the same session id across calls is what lets the agent "remember" earlier turns — a new session id starts a blank conversation.

we use 'Sessions' for short term memory management and 'Memory' for long term memory. 

a Session is comprised of two key components 'Events' and 'State'

* SessionService: The storage layer
    * Manages creation, storage, and retrieval of session data
    * Different implementations for different needs (memory, database, cloud)
* Runner: The orchestration layer
    * Manages the flow of information between user and agent
    * Automatically maintains conversation history
    * Handles the Context Engineering behind the scenes

In [ ]:
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

MODEL_NAME = "gemini-2.5-flash-lite"


# Step 1: Create the LLM Agent
root_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot",  # Description of the agent's purpose
)

# Step 2: Set up Session Management
# InMemorySessionService stores conversations in RAM (temporary)
session_service = InMemorySessionService()

# Step 3: Create the Runner
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Stateful agent initialized!")
print(f"   - Application: {APP_NAME}")
print(f"   - User: {USER_ID}")
print(f"   - Using: {session_service.__class__.__name__}")

In [ ]:
# Run a conversation with two queries in the same session
# Notice: Both queries are part of the SAME session, so context is maintained
await run_session(
    runner,
    [
        "Hi, I am Sam! What is the capital of United States?",
        "Hello! What is my name?",  # This time, the agent should remember!
    ],
    "stateful-agentic-session",
)

**Where the memory came from.** No history is stored manually — passing the same `session_id` on both queries means the Runner appended every turn's `Event` to that session, and the full event history is replayed into the model's context on the next call. That's why the agent could recall the user's name from the first query.

| Service                   | Use Case              | Persistence           | Best For              |
|---------------------------|-----------------------|-----------------------|-----------------------|
| InMemorySessionService    | Development & Testing | ❌ Lost on restart     | Quick prototypes      |
| DatabaseSessionService    | Self-managed apps     | ✅ Survives restarts   | Small to medium apps  |
| Agent Engine Sessions     | Production on GCP     | ✅ Fully managed       | Enterprise scale      |


In [ ]:
#DatabaseSessionService
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

MODEL_NAME = "gemini-2.5-flash"
# Step 1: Create the same agent (notice we use LlmAgent this time)
chatbot_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot with persistent memory",
)

# Step 2: Switch to DatabaseSessionService
# SQLite database will be created automatically
APP_NAME = "persistent_chat_app"

db_url = "sqlite+aiosqlite:///my_agent_data.db"

session_service = DatabaseSessionService(db_url=db_url)


# Step 3: Create a new runner with persistent storage
runner = Runner(agent=chatbot_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Upgraded to persistent sessions!")
print(f"   - Database: my_agent_data.db")
print(f"   - Sessions will survive restarts!")

**Swapping the storage backend.** Everything else about the agent stays the same — only `session_service` changes, from `InMemorySessionService` to `DatabaseSessionService` backed by a local SQLite file (`my_agent_data.db`). Because the Runner and helper function only talk to `session_service` through its interface, this swap is a one-line change with no other code affected.

Run this below for first time then restart kernel and run second cell below to check the database memeory working correct 

In [ ]:
await run_session(
    runner,
    ["Hi, I am Sam! What is the capital of the United States?", "Hello! What is my name?"],
    "test-db-session-01",
)

**The persistence test.** The cell above runs once and writes its events to `my_agent_data.db`. After restarting the kernel (which wipes all in-memory Python state, including `session_service` and `runner`) and re-running the setup cells above, the cell below reuses the same `session_id` — if the agent still knows the name, the session truly came from disk, not from memory.

In [ ]:
await run_session(
    runner,
    ["What is my name?"],
    "test-db-session-01",
)

session data is isolated so check code below

In [ ]:
await run_session(
    runner, ["Hello! What is my name?"], "test-db-session-02"
)  # Note, we are using new session name

In [ ]:
import sqlite3

def check_data_in_db():
    with sqlite3.connect("my_agent_data.db") as connection:
        cursor = connection.cursor()
        result = cursor.execute(
            "select app_name, session_id, author, content from events"
        )
        print([_[0] for _ in result.description])
        for each in result.fetchall():
            print(each)


check_data_in_db()

**Looking under the hood.** `check_data_in_db` bypasses ADK entirely and queries the SQLite `events` table directly, showing that each user message and model reply is stored as its own row keyed by `app_name` and `session_id`. This is the raw data that `DatabaseSessionService` reconstructs into a session's event history.

## add feature of compaction to have more efficient chatbot 

In [ ]:
# Re-define our app with Events Compaction enabled
research_app_compacting = App(
    name="research_app_compacting",
    root_agent=chatbot_agent,
    # This is the new part!
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,  # Trigger compaction every 3 invocations
        overlap_size=1,  # Keep 1 previous turn for context
    ),
)

db_url = "sqlite+aiosqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

# Create a new runner for our upgraded app
research_runner_compacting = Runner(
    app=research_app_compacting, session_service=session_service
)


print("✅ Research App upgraded with Events Compaction!")

**What compaction does.** As a conversation grows, replaying every past event into the model on every turn gets expensive. `EventsCompactionConfig` periodically collapses older events into a short LLM-generated summary, so future turns send that summary plus only the most recent events instead of the full history. `compaction_interval=3` triggers this every 3 invocations, and `overlap_size=1` keeps one turn of raw context around the boundary so nothing feels abruptly cut off.

In [ ]:
# Turn 1
await run_session(
    research_runner_compacting,
    "What is the latest news about AI in healthcare?",
    "compaction_demo",
)

# Turn 2
await run_session(
    research_runner_compacting,
    "Are there any new developments in drug discovery?",
    "compaction_demo",
)



In [ ]:
# Turn 3 - Compaction should trigger after this turn!
await run_session(
    research_runner_compacting,
    "Tell me more about the second development you found.",
    "compaction_demo",
)


With 3 turns now sent to `compaction_demo`, compaction should have fired. The next cell scans the session's events for one with a populated `actions.compaction`, which marks the point where older turns were replaced by a summary.

In [ ]:
# Get the final session state
final_session = await session_service.get_session(
    app_name=research_runner_compacting.app_name,
    user_id=USER_ID,
    session_id="compaction_demo",
)

print("--- Searching for Compaction Summary Event ---")
found_summary = False
for event in final_session.events:
    # Compaction events have a 'compaction' attribute
    if event.actions and event.actions.compaction:
        print("\n✅ SUCCESS! Found the Compaction Event:")
        print(f"  Author: {event.author}")
        print(f"\n Compacted information: {event}")
        found_summary = True
        break

if not found_summary:
    print(
        "\n❌ No compaction event found. Try increasing the number of turns in the demo."
    )

## adding tools

In [ ]:
# Define scope levels for state keys (following best practices)
USER_NAME_SCOPE_LEVELS = ("temp", "user", "app")


# This demonstrates how tools can write to session state using tool_context.
# The 'user:' prefix indicates this is user-specific data.
def save_userinfo(
    tool_context: ToolContext, user_name: str, country: str
) -> Dict[str, Any]:
    """
    Tool to record and save user name and country in session state.

    Args:
        user_name: The username to store in session state
        country: The name of the user's country
    """
    # Write to session state using the 'user:' prefix for user data
    tool_context.state["user:name"] = user_name
    tool_context.state["user:country"] = country

    return {"status": "success"}


# This demonstrates how tools can read from session state.
def retrieve_userinfo(tool_context: ToolContext) -> Dict[str, Any]:
    """
    Tool to retrieve user name and country from session state.
    """
    # Read from session state
    user_name = tool_context.state.get("user:name", "Username not found")
    country = tool_context.state.get("user:country", "Country not found")

    return {"status": "success", "user_name": user_name, "country": country}


print("✅ Tools created.")

**Tools reading and writing state.** `save_userinfo` and `retrieve_userinfo` use `tool_context.state`, a dict-like view onto the session's state, to persist and recall data across turns. The `user:` prefix is an ADK convention marking a key as scoped to the user (as opposed to `temp:` for one-off scratch values or `app:` for app-wide data) — see `USER_NAME_SCOPE_LEVELS` above.

In [ ]:
# Configuration
APP_NAME = "default"
USER_ID = "default"
MODEL_NAME = "gemini-2.5-flash-lite"

# Create an agent with session state tools
root_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="""A text chatbot.
    Tools for managing user context:
    * To record username and country when provided use `save_userinfo` tool. 
    * To fetch username and country when required use `retrieve_userinfo` tool.
    """,
    tools=[save_userinfo, retrieve_userinfo],  # Provide the tools to the agent
)

# Set up session service and runner
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, session_service=session_service, app_name="default")

print("✅ Agent with session state tools initialized!")

In [ ]:
# Test conversation demonstrating session state
await run_session(
    runner,
    [
        "Hi there, how are you doing today? What is my name?",  # Agent shouldn't know the name yet
        "My name is Sam. I'm from Poland.",  # Provide name - agent should save it
        "What is my name? Which country am I from?",  # Agent should recall from session state
    ],
    "state-demo-session",
)

**Confirming it worked.** The transcript above shows the tool calls happening implicitly — the agent invokes `save_userinfo` when told the name/country, then `retrieve_userinfo` when asked again. The cell below inspects `session.state` directly to show the `user:name` and `user:country` keys the tool wrote, proving the data is stored as structured state rather than just re-read from the conversation text.

In [ ]:
# Retrieve the session and inspect its state
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="state-demo-session"
)

print("Session State Contents:")
print(session.state)
print("\n🔍 Notice the 'user:name' and 'user:country' keys storing our data!")